# NEMU walkthrough
## Notice · Explain · Match · Uplift

This notebook is the demo script for a **synthetic** American Express prototype.
No issuer data was used. We generated a world in which we know, dollar for dollar,
how much cross-border spend leaked because the card was not accepted — then we
**hid that file** from the estimator and asked it to find the money anyway.

The centrepiece result is computed in the next cell from `outputs/notice_metrics.json`,
which is produced by `run_pipeline.py` without ever putting `ground_truth.csv` in the
PPML likelihood.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "outputs").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
OUT = ROOT / "outputs"

metrics = json.loads((OUT / "notice_metrics.json").read_text())
holdout = json.loads((OUT / "holdout_metrics.json").read_text())

hidden = metrics["hidden_acceptance_leakage"]
found = metrics["estimated_leakage"]
cause_acc = metrics["cause_value_weighted_accuracy"]

print("=" * 72)
print(
    f"We hid ${hidden:,.0f} of leakage. "
    f"The model found ${found:,.0f} "
    f"({found / hidden:.0%}) and attributed "
    f"{cause_acc:.0%} of it to the correct cause."
)
print("=" * 72)
print(f"corridor correlation vs hidden labels: {metrics['corridor_corr_vs_acceptance_leakage']:.3f}")
print(f"corridor MAPE:                         {metrics['corridor_mape_vs_acceptance_leakage']:.1%}")
print(f"holdout calibration (realized/pred):   {holdout['calibration_ratio']:.2f}")


## 1. The data-generating process (what we hid)

Each trip × category has a *true* spend that does not care whether Amex is accepted.
Observed spend is that demand, suppressed by two independent channels:

\\[
\text{observed} = \text{true} \cdot f(\text{acceptance density}) \cdot g(\text{cash intensity}) \cdot \varepsilon
\\]

- \(f\) is a **sigmoid** of district-level acceptance — sparse districts leak nonlinearly.
- \(g\) is a **country-level cash drain** — the confound. Vietnam is both uncovered *and* cash-heavy.
- `ground_truth.csv` stores `acceptance_leakage`, the quantity Notice is designed to recover:
  counterfactual spend at acceptance \(= 0.95\), holding cash fixed, minus observed.


In [ ]:
members = pd.read_csv(OUT / "members.csv")
trips = pd.read_csv(OUT / "trips.csv")
txns = pd.read_csv(OUT / "transactions.csv")
truth = pd.read_csv(OUT / "ground_truth.csv")

print(f"{len(members):,} members  |  {len(trips):,} trips  |  {len(txns):,} observed txns")
print(f"true spend        ${truth.true_spend.sum():,.0f}")
print(f"observed spend    ${truth.observed_spend.sum():,.0f}")
print(f"total gap         ${truth.total_leakage.sum():,.0f}   (acceptance + cash)")
print(f"acceptance gap    ${truth.acceptance_leakage.sum():,.0f}   ← hidden estimand")
print()
print("oracle cause mix")
print(truth.true_cause.value_counts(normalize=True).mul(100).round(1))


## 2. Notice — PPML gravity, presence reconstructed

**Presence reconstruction.** `trips.csv` tells us the member was in the district even
when `transactions.csv` is silent. Dropping the zeros would throw away the extensive
margin.

**Identification.** Acceptance varies at the *district* level. Destination-*country*
fixed effects absorb cash, prices, and tastes. The log-acceptance coefficients are
identified from Ubud vs SCBD, Khao San vs Sukhumvit — not from "Vietnam is poorer
than Japan".

**Estimand.** \(\widehat{\text{leakage}} = \hat E[\text{spend} \mid a=0.95, x] - \hat E[\text{spend} \mid a=a_0, x]\).
We do **not** subtract raw observed spend: that would dump the residual into "coverage
leakage" and invent a problem in Ginza.


In [ ]:
est = pd.read_csv(OUT / "leakage_estimates.csv")
corridor = pd.read_csv(OUT / "validation_corridor.csv")

print(open(OUT / "ppml_summary.txt").read().split("Pred. interval")[0][-1800:])

fig, ax = plt.subplots(figsize=(6.2, 6.2))
ax.scatter(corridor.truth_acc, corridor.est, c="#1B365D", alpha=0.85)
lim = max(corridor.truth_acc.max(), corridor.est.max()) * 1.05
ax.plot([0, lim], [0, lim], "--", color="#5C6B73", label="y = x")
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("Hidden acceptance leakage (USD)")
ax.set_ylabel("PPML estimate (USD)")
ax.set_title("Corridor validation — model never saw the x-axis")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "validation_scatter.png", dpi=140)
print("wrote outputs/validation_scatter.png")


In [ ]:
rank = (
    est.groupby("dest_country", as_index=False)
    .agg(observed=("observed_spend", "sum"), recoverable=("leakage_estimate", "sum"))
)
rank["rank_obs"] = rank.observed.rank(ascending=False).astype(int)
rank["rank_rec"] = rank.recoverable.rank(ascending=False).astype(int)
rank["shift"] = rank.rank_obs - rank.rank_rec
print(rank.sort_values("recoverable", ascending=False).to_string(index=False))
print()
print("Positive shift = a market the captured-spend dashboard would under-rank.")


## 3. Explain — which lever

| cause | action |
|---|---|
| `no_acceptance` | send merchant acquiring (Match / clustering) |
| `rail_substitution` | send an offer (Match / uplift) |
| `cash` | do not confuse a cash culture with a coverage hole |
| `no_demand` | do nothing |

Rules are thresholded on acceptance density, cash intensity, and the extensive
margin. A PU stretch uses simulated declined auths as the only confirmed
positives for `no_acceptance` (Elkan–Noto). Thresholds were **not** tuned on
`true_cause`.


In [ ]:
print("predicted cause mix")
print(est.cause.value_counts(normalize=True).mul(100).round(1))
print()
print(f"row accuracy:              {metrics['cause_row_accuracy']:.1%}")
print(f"leakage-weighted accuracy: {metrics['cause_value_weighted_accuracy']:.1%}")

cm = (
    est.merge(truth[["trip_id", "category", "true_cause"]], on=["trip_id", "category"])
    .pivot_table(index="true_cause", columns="cause", values="trip_id", aggfunc="count", fill_value=0)
)
print()
print(cm)


## 4. Match — sign merchants, or send an offer

**Acquiring.** `no_acceptance` rows are clustered by district and ranked by
recoverable value. A greedy algorithm adds one merchant at a time to the district
with the highest *marginal* covered value. Coverage \(1 - e^{-\lambda n}\) is
monotone submodular, so greedy is \((1-1/e)\)-optimal.

**Incentives.** `rail_substitution` members get a three-arm policy
(`no_offer`, `2x_points`, `statement_credit`). We simulate randomized assignment
from a known CATE, recover it with CausalForestDML (fallback: X-learner), and
recommend the arm that maximises
\(\text{uplift} \times \text{spend} \times \text{take-rate} - \text{incentive cost}\).


In [ ]:
merch = pd.read_csv(OUT / "merchant_targets.csv")
show = merch[merch.merchants_signed > 0].sort_values("recoverable_value", ascending=False)
print(show[[
    "rank_by_value", "dest_country", "dest_district", "acceptance_density",
    "recoverable_value", "merchants_signed", "coverage", "covered_value",
]].round(2).to_string(index=False))
print()
upl = pd.read_csv(OUT / "uplift_recommendations.csv")
print("recommended arm mix")
print(upl.recommended_arm.value_counts().to_string())
print(f"expected net value of policy: ${upl.expected_net_value.sum():,.0f}")
print(f"CATE corr 2x_points vs oracle:          {metrics['uplift_corr_2x']:.3f}")
print(f"CATE corr statement_credit vs oracle:   {metrics['uplift_corr_statement_credit']:.3f}")


## 5. Uplift — freeze the policy, randomize a holdout

Eligible members (recommended arm ≠ `no_offer`) are split 50/50. Treatment
receives the recommended offer; control receives nothing. Incremental spend is
the difference in realized category spend, with a bootstrap CI. Calibration is
realized incremental / predicted incremental — a finance partner's question.


In [ ]:
print(f"T − C mean spend:     ${holdout['mean_spend_diff_treatment_minus_control']:,.2f}")
print(f"95% bootstrap CI:     [${holdout['spend_diff_ci_low']:,.2f}, ${holdout['spend_diff_ci_high']:,.2f}]")
print(f"predicted $/treated:  ${holdout['predicted_incremental_per_treated']:,.2f}")
print(f"realized $/treated:   ${holdout['realized_incremental_per_treated_net_of_control']:,.2f}")
print(f"calibration ratio:    {holdout['calibration_ratio']:.2f}")

calib = pd.read_csv(OUT / "holdout_calibration.csv")
print()
print(calib.round(2).to_string(index=False))


## How to re-run

```bash
.venv/bin/python run_pipeline.py
.venv/bin/streamlit run dashboard/app.py
```

The dashboard Validation tab is the same scatter as above, built for a live room.
